# Supplementary Figure 12.4: Latent interpolations with NucleusNet-10K.

NucleusNet-10K was encoded to latent vectors and random pairs of vectors were interpolated and reconstructed.

In [1]:
#| label: sfig12d_data

%matplotlib widget

import os
import random
import logging
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display
from pathlib import Path

# Quiet logs
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TQDM_DISABLE"] = "1"
for _name in ("fsspec", "huggingface_hub", "urllib3", "datasets"):
    logging.getLogger(_name).setLevel(logging.ERROR)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, Conv2DTranspose, AveragePooling2D,
    Flatten, Dense, Reshape, ReLU
)

# ── Config ──────────────────────────────────────────────────────────────
WEIGHT_DIR = Path("..") / "data" / "nucleus-ae"
DEC_NAME   = "decoder_weights.h5"
VECTORS_NAME = "nucleusnet10k_encoded_vectors.npy"
LATENT_DIM = 512

T_VALUES  = np.linspace(0.0, 1.0, 6)

RANDOM_SEED = 42

# ── Build decoder ───────────────────────────────────────────────────────
def build_decoder(latent_dim=LATENT_DIM):
    z_in = Input((latent_dim,), name="z_sampling")
    x = Dense(16 * 16 * 128)(z_in)
    x = Reshape((16, 16, 128))(x)
    for filters in [128, 64, 32, 16]:
        x = Conv2DTranspose(filters, 3, strides=2, padding="same")(x)
        x = ReLU()(x)
        x = Conv2D(filters, 3, padding="same")(x)
        x = ReLU()(x)
    out = Conv2D(1, 3, padding="same", activation="sigmoid", name="decoder_output")(x)
    return Model(z_in, out, name="decoder")

# GPU memory growth
try:
    for g in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(g, True)
except Exception:
    pass

# ── Load decoder and encoded vectors ────────────────────────────────────
decoder = build_decoder()

dec_path = WEIGHT_DIR / DEC_NAME
vectors_path = WEIGHT_DIR / VECTORS_NAME

if dec_path.exists():
    decoder.load_weights(str(dec_path))

# Load pre-encoded vectors
if not vectors_path.exists():
    raise FileNotFoundError(f"Encoded vectors not found at {vectors_path}")

encoded_vectors = np.load(str(vectors_path))
N_VECTORS = len(encoded_vectors)

# ── Decoding ────────────────────────────────────────────────────────────
def decode_many(z_batch):
    rec = decoder.predict(z_batch, batch_size=min(len(z_batch), 16), verbose=0)[..., 0]
    return np.clip(rec, 0.0, 1.0)

# ── Pair generation ─────────────────────────────────────────────────────
rng_py = random.Random(RANDOM_SEED)

def generate_random_pair():
    """Generate a single random pair of vector indices."""
    if N_VECTORS < 2:
        return None
    
    a = rng_py.randrange(N_VECTORS)
    b = rng_py.randrange(N_VECTORS)
    while b == a:  # Ensure different indices
        b = rng_py.randrange(N_VECTORS)
    
    return (a, b)

# ── Figure setup ────────────────────────────────────────────────────────
_was_interactive = plt.isinteractive()
plt.ioff()

K = len(T_VALUES)
fig = plt.figure(figsize=(4.4, 2.0), dpi=150, constrained_layout=False)
gs = fig.add_gridspec(1, K)

axes, ims = [], []
for k in range(K):
    ax = fig.add_subplot(gs[0, k])
    im = ax.imshow(np.zeros((256, 256)), cmap="gray", vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)
    ax.set_title(f"t={T_VALUES[k]:.2f}", fontsize=6, pad=2)
    axes.append(ax)
    ims.append(im)

fig.subplots_adjust(left=0.01, right=0.99, bottom=0.02, top=0.88, wspace=0.02)
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.layout.width = "auto"
fig.canvas.layout.height = "auto"

if _was_interactive:
    plt.ion()

# ── State & display ────────────────────────────────────────────────────

def show_frames(frames):
    for k, im in enumerate(ims):
        im.set_data(frames[k])
    fig.canvas.draw_idle()

def compute_and_show_pair():
    """Generate a random pair and display interpolation."""
    pair = generate_random_pair()
    if pair is None:
        return
    
    a_idx, b_idx = pair
    
    # Get pre-encoded vectors
    A_z = encoded_vectors[a_idx]
    B_z = encoded_vectors[b_idx]
    
    # Interpolate in latent space
    Z = np.stack([(1.0 - t) * A_z + t * B_z for t in T_VALUES], axis=0)
    frames = decode_many(Z)
    
    show_frames(frames)

# ── Widgets ─────────────────────────────────────────────────────────────
w_resample = W.Button(description="Resample", button_style="primary")

def on_resample_clicked(_):
    compute_and_show_pair()

w_resample.on_click(on_resample_clicked)

# ── Display ─────────────────────────────────────────────────────────────
container = W.VBox([
    w_resample,
    fig.canvas
])
display(container)

# Initial load
compute_and_show_pair()